# White-Box Audit Continuation (Shadows 8-11)

**Purpose:** Resume white-box audit from shadow 8 after interruption

**Strategy:** 
- Load existing checkpoint from shadow 8
- Train shadows 8, 9, 10, 11
- Save progress after each shadow (safe checkpointing)
- Allow seamless continuation for shadows 12-15 in next session

In [1]:
# ============================================================
# 1.  Setup and Imports
# ============================================================

import os
import sys
import math
import pickle
import json
import gc
import time
import random
import warnings
from pathlib import Path
from datetime import datetime
from copy import deepcopy

import numpy as np
import scipy.stats
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

warnings.filterwarnings("ignore")

# GPU setup
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU:    NVIDIA L40S
Memory: 47.7 GB


In [2]:
# ============================================================
# 2.  Load Previous Configuration and Models
# ============================================================

# Paths (must match original notebook)
EXP_DIR = Path("project/experiments")
AUDIT_DIR = EXP_DIR / "audit"
WB_CKPT_DIR = AUDIT_DIR / "wb_shadows"
WB_RESULTS_FILE = AUDIT_DIR / "wb_results.pkl"

# Create directories if needed
WB_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration from the results file if it exists
if WB_RESULTS_FILE.exists():
    with open(WB_RESULTS_FILE, "rb") as f:
        saved_state = pickle.load(f)
    completed_shadows = saved_state.get("completed_shadows", 0)
    total_shadows = saved_state.get("total_shadows", 16)
    print(f"✓ Loaded checkpoint: {completed_shadows}/{total_shadows} shadows completed")
else:
    print("✗ No checkpoint found! Run original notebook first.")
    sys.exit(1)

# Load the original configuration (these should match your CFG from original notebook)
CFG = saved_state.get("cfg", {})
if not CFG:
    # Fallback configuration (must match original)
    CFG = {
        "num_canaries": 500,
        "num_shadows": 16,
        "delta": 1e-5,
        "audit_epochs": 8,
        "audit_batch": 64,
        "max_grad_norm": 1.0,
        "wb_score_every": 5,
        "confidence_p": 0.05,
        "seed": 2024,
        "exp_dir": EXP_DIR,
        "canary_type": "random_labels",
    }

print(f"\nConfiguration:")
print(f"  Total shadows: {CFG['num_shadows']}")
print(f"  Completed: {completed_shadows}")
print(f"  Canaries: {CFG['num_canaries']}")
print(f"  Epochs per shadow: {CFG['audit_epochs']}")

✓ Loaded checkpoint: 13/16 shadows completed

Configuration:
  Total shadows: 16
  Completed: 13
  Canaries: 500
  Epochs per shadow: 8


In [3]:
# ============================================================
# 3.  Recreate Data Generator (from original notebook)
# ============================================================

# We need the original dataset and transforms
class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        for img_path in sorted(self.root_dir.rglob("*.jpg")) + sorted(self.root_dir.rglob("*.JPG")):
            try:
                stem = img_path.stem.replace(".chip", "")
                parts = stem.split("_")
                if len(parts) >= 2:
                    age = int(parts[0])
                    if 1 <= age <= 116:
                        self.samples.append({"path": str(img_path), "age": age})
            except Exception:
                pass
        print(f"UTKFace: {len(self.samples)} samples loaded.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(s["age"], dtype=torch.float32)

from PIL import Image

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load full dataset
full_ds = UTKFaceDataset(Path("project/data/UTKFace"), transform=None)
N = len(full_ds)
n_val = int(N * 0.15)
perm = np.random.permutation(N)
val_idx = perm[:n_val].tolist()
train_idx = perm[n_val:].tolist()

class _TransformSubset(Dataset):
    def __init__(self, base_ds, indices, transform):
        self.base = base_ds
        self.idx = indices
        self.tf = transform
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        img, age = self.base[self.idx[i]]
        return self.tf(img), age

train_ds = _TransformSubset(full_ds, train_idx, train_tf)
val_ds = _TransformSubset(full_ds, val_idx, val_tf)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

UTKFace: 33488 samples loaded.
Train: 28465, Val: 5023


In [4]:
# ============================================================
# 4.  Recreate AuditDataGenerator
# ============================================================

class AuditDataGenerator:
    """Manages canary injection and IN/OUT splits for shadow models."""
    def __init__(self, base_ds, n_shadows, num_canaries, canary_type, seed):
        self.base_ds = base_ds
        self.n_shadows = n_shadows
        self.num_canaries = num_canaries
        self.canary_type = canary_type
        self.N = len(base_ds)
        rng = np.random.default_rng(seed)

        # Pick canary indices
        perm = rng.permutation(self.N)
        self.canary_indices = perm[:num_canaries]

        # Global Bernoulli membership matrix
        self.shadow_in = np.ones((self.N, n_shadows), dtype=bool)
        coins = rng.integers(0, 2, size=(num_canaries, n_shadows)).astype(bool)
        self.shadow_in[self.canary_indices, :] = coins

        # Canary labels
        self.canary_ages = rng.integers(1, 81, size=num_canaries).astype(np.float32)

    def build_shadow_dataset(self, shadow_idx):
        in_mask = self.shadow_in[:, shadow_idx]
        in_indices = np.where(in_mask)[0].tolist()

        class _ShadowDS(Dataset):
            def __init__(self2, base, indices, canary_indices, canary_ages, canary_type):
                self2.base = base
                self2.indices = indices
                self2.canary_set = set(canary_indices.tolist())
                self2.canary_map = {int(ci): float(ca) for ci, ca in zip(canary_indices, canary_ages)}
                self2.canary_type = canary_type
            def __len__(self2):
                return len(self2.indices)
            def __getitem__(self2, i):
                real_i = self2.indices[i]
                img, age = self2.base[real_i]
                if real_i in self2.canary_set:
                    if self2.canary_type == "random_labels":
                        age = torch.tensor(self2.canary_map[real_i], dtype=torch.float32)
                return img, age

        return _ShadowDS(self.base_ds, in_indices, self.canary_indices, self.canary_ages, self.canary_type)

    def membership_vector(self, shadow_idx):
        return self.shadow_in[self.canary_indices, shadow_idx]


# Create generator with same seed
audit_gen = AuditDataGenerator(
    base_ds=train_ds,
    n_shadows=CFG["num_shadows"],
    num_canaries=CFG["num_canaries"],
    canary_type=CFG["canary_type"],
    seed=CFG["seed"],
)

print("AuditDataGenerator recreated.")

AuditDataGenerator recreated.


In [5]:
# ============================================================
# 5.  Build Canary Tensors
# ============================================================

def build_canary_tensors(audit_gen, cfg):
    m = cfg["num_canaries"]
    imgs_cpu, ages_cpu = [], []
    for c_local in range(m):
        c_global = int(audit_gen.canary_indices[c_local])
        img, age_orig = audit_gen.base_ds[c_global]
        if cfg["canary_type"] == "random_labels":
            age = torch.tensor(audit_gen.canary_ages[c_local], dtype=torch.float32)
        else:
            age = age_orig
        imgs_cpu.append(img)
        ages_cpu.append(age)
    return imgs_cpu, ages_cpu

canary_imgs_cpu, canary_ages_cpu = build_canary_tensors(audit_gen, CFG)
print(f"Built {len(canary_imgs_cpu)} canary tensors")

Built 500 canary tensors


In [6]:
# ============================================================
# 6.  Recreate build_model function
# ============================================================

import torchvision.models as tv_models
from opacus.validators import ModuleValidator

class _GNBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(min(32, planes), planes)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.GroupNorm(min(32, planes), planes)
        self.down = downsample

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + (self.down(x) if self.down else x))

class OpacusResNet18(nn.Module):
    def __init__(self, num_outputs=1):
        super().__init__()
        self.inplanes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.GroupNorm(min(32, 64), 64),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(64, 2, stride=1)
        self.layer2 = self._make(128, 2, stride=2)
        self.layer3 = self._make(256, 2, stride=2)
        self.layer4 = self._make(512, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_outputs)

    def _make(self, planes, blocks, stride):
        down = None
        if stride != 1 or self.inplanes != planes:
            down = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes), planes),
            )
        layers = [_GNBlock(self.inplanes, planes, stride, down)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(_GNBlock(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.pool(x), 1)).squeeze(1)

PRETRAINED_PATH = Path("/home/govindu.srimaan/project/data/resnet18-f37072fd.pth")

def build_model(pretrained=False):
    model = OpacusResNet18(num_outputs=1)
    if pretrained and PRETRAINED_PATH.exists():
        # Load pretrained weights (simplified - we don't need full transfer for shadow models)
        pass
    return model.to(device)

print("Model builder ready.")

Model builder ready.


In [7]:
# ============================================================
# 7.  Core White-Box Audit Function (Resume Capable)
# ============================================================

def score_canaries_loss(model, imgs_cpu, ages_cpu, batch_size=64):
    """Compute per-canary L1 loss."""
    m = len(imgs_cpu)
    crit = nn.L1Loss(reduction="none")
    losses = np.zeros(m, dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, m, batch_size):
            end = min(start + batch_size, m)
            imgs = torch.stack(imgs_cpu[start:end]).to(device)
            ages = torch.stack(ages_cpu[start:end]).to(device)
            preds = model(imgs)
            losses[start:end] = crit(preds, ages).cpu().numpy()
    return losses


def whitebox_audit_continue(audit_gen, cfg, canary_imgs_cpu, canary_ages_cpu, start_shadow, target_shadows):
    """
    Continue white-box audit from start_shadow to target_shadows.
    Saves checkpoint after each shadow.
    """
    m = cfg["num_canaries"]
    S = cfg["num_shadows"]
    epochs = cfg["audit_epochs"]
    batch = cfg["audit_batch"]
    C_clip = cfg["max_grad_norm"]
    score_every = cfg.get("wb_score_every", 5)

    # Load existing results
    # Load existing results
    results_file = cfg["exp_dir"] / "audit" / "wb_results.pkl"
    with open(results_file, "rb") as f:
        saved = pickle.load(f)

    completed = saved["completed_shadows"]
    total_shadows = cfg["num_shadows"]

    # Expand to full size if needed
    if saved["shadow_scores"].shape[0] < total_shadows:
        # Create full-sized arrays
        all_scores = np.zeros((total_shadows, m), dtype=np.float32)
        all_members = np.zeros((total_shadows, m), dtype=bool)
    # Copy completed rows
        all_scores[:completed] = saved["shadow_scores"]
        all_members[:completed] = saved["shadow_members"]
    else:
        all_scores = saved["shadow_scores"]
        all_members = saved["shadow_members"]

    print(f"Loaded {completed}/{total_shadows} shadows. Array shape: {all_scores.shape}")
    
    print(f"\n{'='*60}")
    print(f"RESUMING WHITE-BOX AUDIT")
    print(f"Shadows {start_shadow} to {target_shadows-1}")
    print(f"{'='*60}")

    total_start = time.time()

    for s in range(start_shadow, target_shadows):
        shadow_start = time.time()
        print(f"\n{'='*60}")
        print(f"[WB] Shadow {s:02d}/{S-1}")
        print(f"{'='*60}")

        model = build_model(pretrained=False)
        opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

        shadow_ds = audit_gen.build_shadow_dataset(s)
        loader_s = DataLoader(shadow_ds, batch_size=batch, shuffle=True, num_workers=4, pin_memory=True)

        wb_scores_s = np.zeros(m, dtype=np.float64)
        step_count = 0
        scored_steps = 0

        print("  Moving canaries to GPU...")
        canary_imgs_gpu = [img.to(device) for img in canary_imgs_cpu]
        canary_ages_gpu = [age.to(device) for age in canary_ages_cpu]

        for ep in range(epochs):
            model.train()
            ep_loss = 0.0
            ep_n = 0
            n_batches = len(loader_s)
            print(f"  Epoch {ep+1}/{epochs} - {n_batches} batches")

            for batch_idx, (imgs, ages) in enumerate(loader_s):
                imgs, ages = imgs.to(device), ages.to(device)
                do_score = (step_count % score_every == 0)

                if do_score:
                    w_prev = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

                opt.zero_grad()
                preds = model(imgs)
                loss = nn.L1Loss()(preds, ages)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), C_clip)
                opt.step()
                ep_loss += loss.item() * imgs.size(0)
                ep_n += imgs.size(0)

                if do_score:
                    delta_w = {}
                    for name, p in model.named_parameters():
                        if name in w_prev and p.requires_grad:
                            delta_w[name] = w_prev[name] - p.detach()

                    model.eval()
                    for c in range(m):
                        c_img = canary_imgs_gpu[c]
                        c_age = canary_ages_gpu[c]

                        opt.zero_grad()
                        pred = model(c_img.unsqueeze(0))
                        c_loss = nn.L1Loss()(pred, c_age.unsqueeze(0))
                        c_loss.backward()

                        g_norm_sq = sum(p.grad.norm().item() ** 2 for p in model.parameters() if p.grad is not None)
                        g_norm = np.sqrt(g_norm_sq) if g_norm_sq > 0 else 1e-8
                        clip_coef = min(1.0, C_clip / g_norm)

                        dot = 0.0
                        for name, p in model.named_parameters():
                            if p.grad is not None and name in delta_w:
                                grad_clipped = p.grad * clip_coef
                                dot += (delta_w[name] * grad_clipped).sum().item()
                        wb_scores_s[c] += dot
                        opt.zero_grad()

                    model.train()
                    scored_steps += 1

                step_count += 1

                if (batch_idx + 1) % 100 == 0:
                    print(f"    Batch {batch_idx+1}/{n_batches}, loss={ep_loss/ep_n:.3f}, scored={scored_steps}")

            print(f"    Epoch {ep+1}/{epochs} completed - Avg loss: {ep_loss/ep_n:.3f}, Scored steps: {scored_steps}")

        # Store results
        members_s = audit_gen.membership_vector(s)
        all_scores[s] = wb_scores_s.astype(np.float32)
        all_members[s] = members_s

        in_mask = members_s
        out_mask = ~members_s
        in_mean = wb_scores_s[in_mask].mean() if in_mask.any() else 0
        out_mean = wb_scores_s[out_mask].mean() if out_mask.any() else 0

        shadow_time = time.time() - shadow_start
        print(f"\n  ✓ Shadow {s} completed in {shadow_time:.1f}s")
        print(f"    IN:  n={in_mask.sum():3d}, score={in_mean:+.4f}")
        print(f"    OUT: n={out_mask.sum():3d}, score={out_mean:+.4f}")
        print(f"    Δ = {in_mean - out_mean:+.4f}")

        # Save checkpoint
        with open(results_file, "wb") as f:
            pickle.dump({
                "shadow_scores": all_scores[:s+1],
                "shadow_members": all_members[:s+1],
                "completed_shadows": s + 1,
                "total_shadows": S,
                "cfg": cfg,
                "timestamp": datetime.now().isoformat()
            }, f)
        print(f"  💾 Checkpoint saved (shadow {s+1}/{S})")

        del model
        gc.collect()
        torch.cuda.empty_cache()

    total_time = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"Shadows {start_shadow}-{target_shadows-1} COMPLETED in {total_time/60:.1f} minutes!")
    print(f"{'='*60}")

    return all_scores, all_members

In [ ]:
# ============================================================
# 8.  RUN CONTINUATION (Shadows 8-11)
# ============================================================

START_SHADOW = 12   # Resume from shadow 12
TARGET_SHADOWS = 16  # Train up to shadow 15 (0-indexed, so 12 means shadows 12,13,14,15)

print(f"\n{'#'*60}")
print(f"CONTINUING WHITE-BOX AUDIT")
print(f"From shadow {START_SHADOW} to shadow {TARGET_SHADOWS-1}")
print(f"{'#'*60}")

try:
    all_scores, all_members = whitebox_audit_continue(
        audit_gen, CFG, canary_imgs_cpu, canary_ages_cpu,
        start_shadow=START_SHADOW, target_shadows=TARGET_SHADOWS
    )
    print(f"\n✅ Successfully completed shadows {START_SHADOW}-{TARGET_SHADOWS-1}")
except KeyboardInterrupt:
    print(f"\n⚠ Interrupted! Progress saved. Run again to continue from last completed shadow.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


############################################################
CONTINUING WHITE-BOX AUDIT
From shadow 12 to shadow 15
############################################################
Loaded 13/16 shadows. Array shape: (16, 500)

RESUMING WHITE-BOX AUDIT
Shadows 12 to 15

[WB] Shadow 12/15
  Moving canaries to GPU...
  Epoch 1/8 - 441 batches


In [9]:
# ============================================================
# 9.  Verify Checkpoint Status
# ============================================================

# Check current progress
if WB_RESULTS_FILE.exists():
    with open(WB_RESULTS_FILE, "rb") as f:
        saved = pickle.load(f)
    completed = saved.get("completed_shadows", 0)
    total = saved.get("total_shadows", 16)
    print(f"\n{'='*60}")
    print(f"CHECKPOINT STATUS")
    print(f"{'='*60}")
    print(f"Completed shadows: {completed}/{total}")
    print(f"Remaining: {total - completed}")
    
    if completed < total:
        print(f"\n💡 To continue, run this notebook again with:")
        print(f"   START_SHADOW = {completed}")
        print(f"   TARGET_SHADOWS = min({completed + 4}, {total})")
    else:
        print(f"\n✅ All shadows completed! Run epsilon computation.")
else:
    print("\n⚠ No checkpoint found!")


CHECKPOINT STATUS
Completed shadows: 13/16
Remaining: 3

💡 To continue, run this notebook again with:
   START_SHADOW = 13
   TARGET_SHADOWS = min(17, 16)


In [10]:
# ============================================================
# MERGE WHITE-BOX RESULTS FROM BOTH NOTEBOOKS
# ============================================================

import pickle
import numpy as np
from pathlib import Path

# Paths (adjust if your parallel notebook used a different file)
AUDIT_DIR = Path("project/experiments/audit")
WB_RESULTS_MAIN = AUDIT_DIR / "wb_results.pkl"           # Original notebook
WB_RESULTS_PARALLEL = AUDIT_DIR / "wb_results_part2.pkl" # Parallel notebook (if different)

# Alternative: if parallel notebook used same file name but was in different location
# WB_RESULTS_PARALLEL = Path("project/experiments/audit/wb_results_parallel.pkl")

print("="*60)
print("MERGING WHITE-BOX RESULTS")
print("="*60)

# Load main notebook results
if WB_RESULTS_MAIN.exists():
    with open(WB_RESULTS_MAIN, "rb") as f:
        main_results = pickle.load(f)
    main_completed = main_results.get("completed_shadows", 0)
    print(f"Main notebook: {main_completed} shadows completed")
else:
    print("Main notebook results not found!")
    main_results = None
    main_completed = 0

# Load parallel notebook results
if WB_RESULTS_PARALLEL.exists():
    with open(WB_RESULTS_PARALLEL, "rb") as f:
        parallel_results = pickle.load(f)
    parallel_completed = parallel_results.get("completed_shadows", 0)
    print(f"Parallel notebook: {parallel_completed} shadows completed")
else:
    print("Parallel notebook results not found!")
    print("Checking alternative locations...")
    
    # Check if results are in the same file but different format
    if WB_RESULTS_MAIN.exists():
        # The parallel run might have overwritten? Let's check the timestamp
        import time
        mtime = WB_RESULTS_MAIN.stat().st_mtime
        print(f"Main results file modified: {time.ctime(mtime)}")
    
    parallel_results = None
    parallel_completed = 0

# Determine what we have
total_shadows = 16
m = 500  # num_canaries

# Create merged arrays
merged_scores = np.zeros((total_shadows, m), dtype=np.float32)
merged_members = np.zeros((total_shadows, m), dtype=bool)

# Track which shadows we have
shadows_found = []

# Copy from main results
if main_results is not None:
    main_scores = main_results.get("shadow_scores")
    main_members = main_results.get("shadow_members")
    if main_scores is not None:
        n_main = main_scores.shape[0]
        merged_scores[:n_main] = main_scores
        merged_members[:n_main] = main_members
        shadows_found.extend(range(n_main))
        print(f"  Copied shadows 0-{n_main-1} from main")

# Copy from parallel results
if parallel_results is not None:
    parallel_scores = parallel_results.get("shadow_scores")
    parallel_members = parallel_results.get("shadow_members")
    if parallel_scores is not None:
        n_parallel = parallel_scores.shape[0]
        # Parallel likely started at shadow 12? Let's check
        # If parallel has fewer rows, it might be partial
        if n_parallel <= total_shadows:
            merged_scores[:n_parallel] = np.maximum(merged_scores[:n_parallel], parallel_scores)
            merged_members[:n_parallel] = merged_members[:n_parallel] | parallel_members
            shadows_found.extend(range(n_parallel))
            print(f"  Merged shadows 0-{n_parallel-1} from parallel")

# Remove duplicates
shadows_found = sorted(set(shadows_found))
completed = len(shadows_found)

print(f"\n{'='*60}")
print(f"MERGED RESULT: {completed}/{total_shadows} shadows available")
print(f"Shadows present: {shadows_found}")
print(f"Missing shadows: {[i for i in range(total_shadows) if i not in shadows_found]}")
print(f"{'='*60}")

# Save merged results
if completed > 0:
    merged_file = AUDIT_DIR / "wb_results_merged.pkl"
    with open(merged_file, "wb") as f:
        pickle.dump({
            "shadow_scores": merged_scores[:completed],
            "shadow_members": merged_members[:completed],
            "completed_shadows": completed,
            "total_shadows": total_shadows,
            "shadows_indices": shadows_found,
            "timestamp": datetime.now().isoformat()
        }, f)
    print(f"\n✓ Merged results saved to {merged_file}")
    
    # Also save as wb_results.pkl for compatibility
    with open(AUDIT_DIR / "wb_results.pkl", "wb") as f:
        pickle.dump({
            "shadow_scores": merged_scores[:completed],
            "shadow_members": merged_members[:completed],
            "completed_shadows": completed,
            "total_shadows": total_shadows,
            "timestamp": datetime.now().isoformat()
        }, f)
    print(f"✓ Updated main results file with merged data")

MERGING WHITE-BOX RESULTS
Main notebook: 13 shadows completed
Parallel notebook results not found!
Checking alternative locations...
Main results file modified: Tue Jun  9 02:40:42 2026
  Copied shadows 0-12 from main

MERGED RESULT: 13/16 shadows available
Shadows present: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Missing shadows: [13, 14, 15]

✓ Merged results saved to project/experiments/audit/wb_results_merged.pkl
✓ Updated main results file with merged data


In [11]:
# ============================================================
# 10.  Compute Final Epsilon (When All Shadows Complete)
# ============================================================

# Only run this when all shadows are complete

def compute_final_epsilon(results_file, cfg):
    """Compute epsilon bound from completed audit."""
    from scipy.stats import binom
    
    with open(results_file, "rb") as f:
        saved = pickle.load(f)
    
    all_scores = saved["shadow_scores"]
    all_members = saved["shadow_members"]
    
    flat_scores = all_scores.ravel()
    flat_members = all_members.ravel()
    M_total = len(flat_scores)
    
    order = np.argsort(flat_scores)
    sorted_members = flat_members[order].astype(np.int32)
    cumsum_forward = np.cumsum(sorted_members)
    
    max_k = min(500, M_total // 4)
    best_eps, best_k, best_v, best_r = 0.0, 0, 0, 0
    
    def get_eps(m, r, v, delta, confidence=0.05):
        if v <= r // 2:
            return 0.0
        
        def p_value(eps_test):
            q = np.exp(eps_test) / (np.exp(eps_test) + 1)
            if v > r:
                return 1.0
            try:
                p_val = 1.0 - binom.cdf(v - 1, r, q)
            except:
                p_val = 1.0
            if v > 0:
                p_val += 2.0 * m * delta / v
            else:
                p_val += 2.0 * m * delta
            return min(p_val, 1.0)
        
        eps_low, eps_high = 0.0, 10.0
        for _ in range(10):
            if p_value(eps_high) < confidence:
                break
            eps_high *= 2
        for _ in range(40):
            eps_mid = (eps_low + eps_high) / 2
            if p_value(eps_mid) < confidence:
                eps_low = eps_mid
            else:
                eps_high = eps_mid
        return eps_low
    
    print(f"\nSweeping k for optimal ε lower bound...")
    print(f"Total samples: {M_total:,}")
    
    step = max(1, max_k // 50)
    for k in range(1, max_k + 1, step):
        top_in = sorted_members[-k:].sum()
        bot_out = k - cumsum_forward[k-1]
        v_k = int(top_in + bot_out)
        r_k = 2 * k
        eps_k = get_eps(M_total, r_k, v_k, cfg["delta"], cfg.get("confidence_p", 0.05))
        if eps_k > best_eps:
            best_eps, best_k, best_v, best_r = eps_k, k, v_k, r_k
            print(f"  k={k:4d}  v={v_k}/{r_k} ({100*v_k/r_k:.1f}%)  ε≥{best_eps:.4f}  ← new best")
    
    print(f"\n{'='*60}")
    print("WHITE-BOX AUDIT FINAL RESULTS")
    print(f"{'='*60}")
    print(f"ε ≥ {best_eps:.4f}")
    print(f"Optimal: k={best_k}, {best_v}/{best_r} correct ({100*best_v/best_r:.1f}%)")
    
    return best_eps, best_k, best_v, best_r


# Check if all shadows are complete
if WB_RESULTS_FILE.exists():
    with open(WB_RESULTS_FILE, "rb") as f:
        saved = pickle.load(f)
    completed = saved.get("completed_shadows", 0)
    total = saved.get("total_shadows", 16)
    
    if completed >= total:
        print("\n" + "="*60)
        print("ALL SHADOWS COMPLETE! Computing final epsilon...")
        print("="*60)
        final_eps, best_k, best_v, best_r = compute_final_epsilon(WB_RESULTS_FILE, CFG)
    else:
        print(f"\n⚠ Not all shadows complete: {completed}/{total}")
        print(f"Continue training more shadows first.")


⚠ Not all shadows complete: 13/16
Continue training more shadows first.
